# 08. Decision Tree

Split on feature questions. Scaling is optional (`scale=False`).
Tune `max_depth` with stratified CV (F1); deeper trees can overfit.

In [ ]:
import sys
sys.path.append("..")

from sklearn.tree import DecisionTreeClassifier

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [5]:
candidates = [3, 5, 8, 12, 15, 20, None]
selection_metric = "f1"

cv_by_depth = {}
best_depth = None
best_score = -1.0

for depth in candidates:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    pipe = make_model_pipeline(model, scale=False)
    cv = cross_val_scores(pipe, X, y)
    cv_by_depth[depth] = cv
    score = cv[selection_metric]["mean"]
    print(f"max_depth={depth}  CV {selection_metric}={score:.4f}  (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_depth = depth

print("selected max_depth:", best_depth)

max_depth=3  CV f1=0.6132  (±0.0050)
max_depth=5  CV f1=0.6200  (±0.0057)
max_depth=8  CV f1=0.6448  (±0.0060)
max_depth=12  CV f1=0.6621  (±0.0045)
max_depth=15  CV f1=0.6668  (±0.0137)
max_depth=20  CV f1=0.6591  (±0.0125)
max_depth=None  CV f1=0.6204  (±0.0075)
selected max_depth: 15


In [6]:
final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

cv = cv_by_depth[best_depth]
pipe2 = make_model_pipeline(final_model, scale=False)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

depth_tag = 'none' if best_depth is None else str(best_depth)
path = save_results(
    f"decision_tree_depth{depth_tag}",
    cv,
    test,
    extra={
        "max_depth": best_depth,
        "scale": False,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_depth": {
            str(d): {m: v["mean"] for m, v in summary.items()}
            for d, summary in cv_by_depth.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)

Test: {'accuracy': 0.8501320557705301, 'balanced_accuracy': 0.7773426184333261, 'f1': 0.66838814895352, 'roc_auc': 0.8692333885554859, 'average_precision': 0.6885350330207575}
Confusion matrix: [[11382, 1053], [1387, 2459]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/decision_tree_depth15.json
